In [ ]:
# Organize Function (equivalent of dplyr's arrange function). From https://stackoverflow.com/questions/75667332/base-analogue-of-arrange-in-pipelines
organize <- function (df, ..., na.last = TRUE, decreasing = FALSE,
                      method = c("auto", "shell", "radix"), drop = FALSE) {
  method <- match.arg(method)
  dots <- eval(substitute(alist(...)))
  exprs <- lapply(dots, FUN = \(e) eval(e, df, parent.frame())) 
  arg_ls <- c(exprs,
              na.last = na.last,
              decreasing = decreasing,
              method = method
              )
  df[do.call("order", arg_ls), , drop = drop]
}

## Opening BMI Data

In [ ]:
file <- system( "tar -tzf /sharedscratch/maff1-group/data/ADNI_PhysicalNeuro.tar.gz", intern = TRUE )

In [ ]:
cmd <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_PhysicalNeuro.tar.gz VITALS_04Jun2025.csv"
vitals <- read.csv( pipe(cmd) )

In [ ]:
# Note that there is height data missing in a lot of columns bc it was only recorded once and assumed to stay constant the rest of the time

In [ ]:
vitals_simple <- vitals[, colnames(vitals) %in% c('RID','VISCODE2','VISDATE','VSWEIGHT', 'VSWTUNIT', 'VSHEIGHT', 'VSHTUNIT')]
vitals_simple 

## Opening Biomarker Data

In [ ]:
files <- system( "tar -tzf /sharedscratch/maff1-group/data/ADNI_Biomarker_ROCHE_ELECSYS.tar.gz", intern = TRUE )
files1 <- system( "tar -tzf /sharedscratch/maff1-group/data/ADNI_Biomarker_FUJIREBIO_QUANTERIX.tar.gz", intern = TRUE )

In [ ]:
cmd2 <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Biomarker_ROCHE_ELECSYS.tar.gz UPENNBIOMK_ROCHE_ELECSYS_04Jun2025.csv"
roche <- read.csv( pipe(cmd2) )
head(roche)|>
    organize(by = RID)

In [ ]:
# Roche elecsys has both ABETA42 and PTAU
r_biomarkers_simple <- roche[, colnames(roche) %in% c('RID','VISCODE2','EXAMDATE','ABETA42', 'PTAU')]
r_biomarkers_simple |>
    organize(by=RID)

In [ ]:
cmd <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Biomarker_FUJIREBIO_QUANTERIX.tar.gz UPENN_PLASMA_FUJIREBIO_QUANTERIX_04Jun2025.csv"
quanterix <- read.csv( pipe(cmd) )
head(quanterix) |>
    organize(by = RID)

In [ ]:
# Qanterix has both pTau-217 & AB-42
q_biomarkers_simple <- quanterix[, colnames(quanterix) %in% c('RID','VISCODE2','EXAMDATE','pT217_F', 'AB42_F')]
q_biomarkers_simple |>
    organize(by=RID)

## Opening Lipid Data

In [ ]:
cmd <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Metabolite_NIGHTINGALE.tar.gz ADNINIGHTINGALELONG_DICT_05_07_21_04Jun2025.csv"
lipids <- read.csv( pipe(cmd) )

In [ ]:
cmd2 <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Metabolite_NIGHTINGALE.tar.gz ADNINIGHTINGALELONG_05_24_21_04Jun2025.csv"
lipid_variables <- read.csv( pipe(cmd2) )
lipid_variables <- lipid_variables[, colnames(lipid_variables) %in% c('RID','VISCODE','VISCODE2','EXAMDATE','TOTAL_TG', 'HDL_C', 'GLUCOSE')]

In [ ]:
head(lipid_variables) |>
    organize(by = RID) 

In [ ]:
# Units appear to be in mmol/L when it needs to be in mg/dL (checked units via: https://biobank.ndph.ox.ac.uk/ukb/ukb/docs/nmrm_phase3_companion.pdf)
# Converting glucose to mg/dL below by multiplying current values by 18.018, cholesterol by multiplying by 38.6, and triglycerides by multiplying by 89 (https://www.google.com/search?q=conversion+rate+from+mmol+l+to+mg+dl&oq=conversion+rate+from+mmol%2FL+to+&gs_lcrp=EgZjaHJvbWUqCAgBEAAYFhgeMgYIABBFGDkyCAgBEAAYFhgeMggIAhAAGBYYHjINCAMQABiGAxiABBiKBTINCAQQABiGAxiABBiKBTINCAUQABiGAxiABBiKBTINCAYQABiGAxiABBiKBTIKCAcQABiABBiiBDIKCAgQABiiBBiJBTIKCAkQABiiBBiJBdIBCTEyODg3ajBqMagCALACAA&sourceid=chrome&ie=UTF-8)
lipid_variables_mgdl <- lipid_variables |>
    transform(HDL_C_mgdl = HDL_C * 38.6) |>
    transform(TG_mgdl = TOTAL_TG * 89) |>
    transform(GLUCOSE_mgdl = GLUCOSE * 18.018)
head(lipid_variables_mgdl)

In [ ]:
lipid_variables_mgdl_simple <- lipid_variables_mgdl[, colnames(lipid_variables_mgdl) %in% c('RID','VISCODE2','EXAMDATE','HDL_C_mgdl', 'TG_mgdl', 'GLUCOSE_mgdl')]
lipid_variables_mgdl_simple

## Finding date differences between biomarker and vitals data

In [ ]:
# Merging vitals data (BMI) and biomarkers data where the VISCODES2 are the same
vitals_biomarkers <- merge(vitals_simple,q_biomarkers_simple, by = c("RID", "VISCODE2"))

# Checking where VISDATE (collection date for height and weight) and EXAMDATE (neurobiomarker collection date) don't match
# TRUE = Means dates were the same for both vital and neurobiomarker measurements
vitals_biomarkers_datecheck <- vitals_biomarkers |>
    transform(SAME_DATES = ifelse(VISDATE == EXAMDATE, TRUE, FALSE))
vitals_biomarkers_datecheck |>
    organize(by = RID)

In [ ]:
# Converting Date Columns from chr variables to dates
vitals_biomarkers_datecheck$VISDATE <- as.Date(vitals_biomarkers_datecheck$VISDATE )
vitals_biomarkers_datecheck$EXAMDATE <- as.Date(vitals_biomarkers_datecheck$EXAMDATE)
vitals_biomarkers_datecheck |>
    organize(by = RID)

In [ ]:
date_difference_calculations <- vitals_biomarkers_datecheck |>
    subset(!SAME_DATES) |>
    transform(DIFFERENCE = VISDATE - EXAMDATE)
date_difference_calculations

In [ ]:
as.numeric(date_difference_calculations$DIFFERENCE)

In [ ]:
# How many datapoints were collected under 2 weeks apart, under 1 month apart, and under 1 year apart
cat("The number of datapoints wherein biomarkers and vitals were taken on the same day is:", 
    sum(vitals_biomarkers_datecheck$SAME_DATE == TRUE),
    "\n")
cat("The number of datapoints wherein biomarkers and vitals were taken within 2 weeks of each other is:", 
    sum(abs(date_difference_calculations$DIFFERENCE) <=14, 
        na.rm = TRUE), "\n")
cat("...Between 2 weeks and 1 month:", 
    sum((abs(date_difference_calculations$DIFFERENCE) > 14 &
         abs(date_difference_calculations$DIFFERENCE) <= 31), 
         na.rm = TRUE), "\n")
cat("...Between 1 month and 1 year:", 
    sum((abs(date_difference_calculations$DIFFERENCE) > 31 &
         abs(date_difference_calculations$DIFFERENCE) <= 365), 
         na.rm = TRUE), "\n")
cat("...Over 1 year apart:", 
    sum((abs(date_difference_calculations$DIFFERENCE) > 365), 
         na.rm = TRUE), "\n")
cat("There is 1 data point with missing date data.")

In [ ]:
# Mergining vitals with lipid data
vitals_lipids <- merge(vitals_simple, lipid_variables, by = c("RID", "VISCODE2"))
vitals_lipids |>
    organize(by = RID)

In [ ]:
table(vitals_lipids$VISCODE2)

In [ ]:
table(q_biomarkers_simple$VISCODE2)

In [ ]:
# Looking at VISCODE2 summary
all_data <- merge(vitals_biomarkers,lipid_variables, by = c("RID", "VISCODE2"))
all_data |>
    organize(by=RID)

In [ ]:
table(all_data$VISCODE2)

## Neurobiomarker Data Timeline

### Selecting Roche Elecsys Justification

Because there is limited longitudinal data for quanterix, pivoting to see if roche elecsys dataset, despite not having Tau-217, is more useful bc of more longitudinal data

In [ ]:
table(r_biomarkers_simple$VISCODE2)
# Seems to be more availability of data. 
    #324 At 12 months as opposed to 132 with quanterix
    #541 At 24 months as opposed to 87 with quanterix
    #234 At 48 month as opposed to 72 with quanterix

In [ ]:
r_biomarkers_simple |>
    organize(by = RID)

### Looking at how many individuals had data for more than 1 time point

In [ ]:
r_biomarkers_RID_count <- aggregate(VISCODE2 ~ RID, data = r_biomarkers_simple, length) 
r_biomarkers_RID_count

In [ ]:
r_biomarkers_RID_count |>
    subset(VISCODE2 == 2)
# There are 491 participants in which there are 2 neurobiomarker datapoints

In [ ]:
r_biomarkers_RID_count |>
    subset(VISCODE2 == 3)
# There are 175 participants in which there are 3 or more neurobiomarker datapoints

In [ ]:
r_biomarkers_RID_count |>
    subset(VISCODE2 > 3)
# There are 173 participants in which there are 3 or more neurobiomarker datapoints

### Looking at timepoints bl, m12, m24, m48

In [ ]:
# Keeping VISCODE2 visits that are either bl,  m12, m24, or m48
r_biomarkers_wanted_visit <- r_biomarkers_simple |>
    subset(VISCODE2 %in% c("bl", "m12", "m24", "m48"))
r_biomarkers_wanted_visit

In [ ]:
# Determining how many participants had bl & m12 measurements 
r_biomarkers_bl_m12_visits <- r_biomarkers_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m12"))
r_biomarkers_bl_m12_visits

In [ ]:
r_biomarkers_blORm12_visits <- aggregate(VISCODE2 ~ RID,
                               data = r_biomarkers_bl_m12_visits, length) 
r_biomarkers_blORm12_visits

In [ ]:
r_biomarkers_blANDm12_visits <-r_biomarkers_blORm12_visits |>
    subset(VISCODE2 == 2)
r_biomarkers_blANDm12_visits
# 309 participants have bl & m12 measurements 

In [ ]:
# Determining how many participants had bl & m24 measurements 
r_biomarkers_bl_m24_visits <- r_biomarkers_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m24"))
r_biomarkers_blORm24_visits <- aggregate(VISCODE2 ~ RID,
                               data = r_biomarkers_bl_m24_visits, length) 
r_biomarkers_blANDm24_visits <-r_biomarkers_blORm24_visits |>
    subset(VISCODE2 == 2)
r_biomarkers_blANDm24_visits
# 530 participants have bl & m24 measurements 

In [ ]:
# Determining how many participants had bl & m48 measurements 
r_biomarkers_bl_m48_visits <- r_biomarkers_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m48"))
r_biomarkers_blORm48_visits <- aggregate(VISCODE2 ~ RID,
                               data = r_biomarkers_bl_m48_visits, length) 
r_biomarkers_blANDm48_visits <-r_biomarkers_blORm48_visits |>
    subset(VISCODE2 == 2)
r_biomarkers_blANDm48_visits
# 224 participants have bl & m48 measurements 

In [ ]:
# Determining how many participants had bl, m24, and m48 measurements 
r_biomarkers_bl_m24_m48_visits <- r_biomarkers_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m24", "m48"))
r_biomarkers_blORm24ORm48_visits <- aggregate(VISCODE2 ~ RID,
                               data = r_biomarkers_bl_m24_m48_visits, length) 
r_biomarkers_blANDm24ANDm48_visits <-r_biomarkers_blORm24ORm48_visits |>
    subset(VISCODE2 == 3)
r_biomarkers_blANDm24ANDm48_visits
# 168 participants have bl, m24, and m48 measurements 

In [ ]:
# Determining how many participants had bl, m12, and m24 measurements 
r_biomarkers_bl_m12_m24_visits <- r_biomarkers_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m12", "m24"))
r_biomarkers_blORm12ORm24_visits <- aggregate(VISCODE2 ~ RID,
                               data = r_biomarkers_bl_m12_m24_visits, length) 
r_biomarkers_blANDm12ANDm24_visits <-r_biomarkers_blORm12ORm24_visits |>
    subset(VISCODE2 == 3)
r_biomarkers_blANDm12ANDm24_visits
# 92 participants have bl, m12, and m24 measurements 

### Summary

#### Two time points
309 participants have bl & m12 measurements |
530 participants have bl & m24 measurements |
224 participants have bl & m48 measurements 
#### Three time points
92 participants have bl, m12, and m24 measurements |
168 participants have bl, m24, and m48 measurements 

# Clarifying IR Metric timeline

### Fixing Height Data

In [ ]:
vitals_simple |>
    organize(by = RID)
# -4.0 is equivalent to NA. Height was only recordeed once (at screening usually)

In [ ]:
# Turning -4.0 to NA values so that they can be replaced by the true height value
library(dplyr)
vitals_simple_NA <- vitals_simple |>
    transform(VSHEIGHT = na_if(VSHEIGHT, -4)) |>
    transform(VSHTUNIT = na_if(VSHTUNIT, -4))
vitals_simple_NA |>
    organize(by = RID)

In [ ]:
sc_heights <- vitals_simple_NA |>
  subset(VISCODE2 == "sc", select = c(RID, VSHEIGHT, VSHTUNIT))
sc_heights

In [ ]:
vitals_filled <- merge(vitals_simple_NA,sc_heights,
                       by = "RID",
                       all.x = TRUE,
                       suffixes = c("", "_SC"))
vitals_filled$VSHEIGHT[is.na(vitals_filled$VSHEIGHT)] <- vitals_filled$VSHEIGHT_SC[is.na(vitals_filled$VSHEIGHT)]
vitals_filled$VSHTUNIT[is.na(vitals_filled$VSHTUNIT)] <- vitals_filled$VSHTUNIT_SC[is.na(vitals_filled$VSHTUNIT)]
vitals_filled |>
    organize(by = RID) 

In [ ]:
# Looking for any more missing weight data (negative values)
table(vitals_filled$VSWEIGHT)

In [ ]:
# Looking for any more missing height data (negative values)
table(vitals_filled$VSHEIGHT)

### Calculating BMI

In [ ]:
# Calculating BMI and removing missing values
vitals_data <- vitals_filled |>
    subset(VSWEIGHT > 0) |>
    subset(VSHEIGHT > 0) |>
    transform(WEIGHT_KG = ifelse(VSWTUNIT == 1, VSWEIGHT * 0.453592, VSWEIGHT))|>
    transform (HEIGHT_M = ifelse(VSHTUNIT == 1, VSHEIGHT * 0.0254, VSHEIGHT / 100)) |>
    transform (BMI = WEIGHT_KG/(HEIGHT_M^2)) |>
    organize(by = RID)
vitals_data

In [ ]:
# Simplifying to BMI Table
BMI_data <- vitals_data[, colnames(vitals_data) %in% c('RID','VISCODE2','VISDATE','BMI')]
BMI_data |>
    organize(by = RID)

In [ ]:
write.csv(BMI_data, "015_BMI Data")

### Calculating IR Metrics

In [ ]:
lipids_BMI_data <- merge(BMI_data, lipid_variables_mgdl_simple, by = c("RID", "VISCODE2"))
lipids_BMI_data |>
    organize(by = RID)

In [ ]:
# Calculating IR Metrics and Showing Clean Dataframe
ir_metrics <- lipids_BMI_data |>
    transform(TG_HDL_Ratio = TG_mgdl/HDL_C_mgdl) |>
    transform(TYG = log((TG_mgdl*GLUCOSE_mgdl)/2)) |>
    transform(TYG_BMI = TYG * BMI) |>
    transform(METS_IR = ((log(2*GLUCOSE_mgdl + TG_mgdl)*BMI)/(log(HDL_C_mgdl))))
ir_metrics_simple <- ir_metrics[, colnames(ir_metrics) %in% c('RID','VISCODE2','EXAMDATE','TYG', 'TYG_BMI', 'METS_IR','TG_HDL_Ratio')]
ir_metrics_simple |>
    organize(by = RID)

In [ ]:
write.csv(ir_metrics_simple,"010_All IR Data")

### Timepoints for IR Data

In [ ]:
table(ir_metrics$VISCODE2)

In [ ]:
ir_metrics_wanted_visit <- ir_metrics_simple |>
    subset(VISCODE2 %in% c("bl", "m12", "m24", "m48"))
ir_metrics_wanted_visit |>
    organize(by = RID)

In [ ]:
# Determining how many participants had bl & m12 measurements 
ir_bl_m12 <- ir_metrics_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m12"))
ir_blORm12 <- aggregate(VISCODE2 ~ RID,
                         data = ir_bl_m12, length) 
ir_blANDm12 <-ir_blORm12 |>
    subset(VISCODE2 == 2)
ir_blANDm12
# 1117 participants have bl & m12 measurements 

In [ ]:
# Determining how many participants had bl & m24 measurements 
ir_bl_m24 <- ir_metrics_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m24"))
ir_blORm24 <- aggregate(VISCODE2 ~ RID,
                         data = ir_bl_m24, length) 
ir_blANDm24 <-ir_blORm24 |>
    subset(VISCODE2 == 2)
ir_blANDm24
# 1022 participants have bl & m24 measurements 

In [ ]:
# Determining how many participants had bl & m48 measurements 
ir_bl_m48 <- ir_metrics_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m48"))
ir_blORm48 <- aggregate(VISCODE2 ~ RID,
                         data = ir_bl_m48, length) 
ir_blANDm48 <-ir_blORm48 |>
    subset(VISCODE2 == 2)
ir_blANDm48
# 153 participants have bl & m48 measurements 

In [ ]:
# Determining how many participants had bl & m12 & m24 measurements 
ir_bl_m12_m24 <- ir_metrics_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m12", "m24"))
ir_blORm12ORm24 <- aggregate(VISCODE2 ~ RID,
                        ir_bl_m12_m24, length) 
ir_blANDm12ANDm24 <-ir_blORm12ORm24 |>
    subset(VISCODE2 == 3)
ir_blANDm12ANDm24
# 828 participants have bl & m24 & m48 measurements 

In [ ]:
# Determining how many participants had bl & m12 & m48 measurements 
ir_bl_m24_m48 <- ir_metrics_wanted_visit |>
    subset(VISCODE2 %in% c("bl", "m24", "m48"))
ir_blORm24ORm48 <- aggregate(VISCODE2 ~ RID,
                        ir_bl_m24_m48, length) 
ir_blANDmm24ANDm48 <-ir_blORm24ORm48 |>
    subset(VISCODE2 == 3)
ir_blANDmm24ANDm48
# 91 participants have bl & m24 & m48 measurements 

### Summary

#### Two time points
1117 participants have bl & m12 measurements |
1022 participants have bl & m24 measurements |
153 participants have bl & m48 measurements 
#### Three time points
828 participants have bl, m12, and m24 measurements |
91 participants have bl, m24, and m48 measurements 

# Merging IR and Neurobiomarker Timelines

In [ ]:
ir_metrics_wanted_visit

In [ ]:
r_biomarkers_wanted_visit

In [ ]:
wanted_visits <- merge(r_biomarkers_wanted_visit, ir_metrics_wanted_visit, by = c("RID", "VISCODE2"))
wanted_visits |>
    organize(by = RID)